# ROC Curve Comparison: DeLong Test and Subject-Level Permutation Test

Addresses **Reviewer 1** request for a formal statistical test comparing radiomics vs DL ROC curves
(DeLong 1988, or subject-level permutation test).

**Radiomics predictions** are available from `09_final_pipeline_PAPER.ipynb` (already run).

**DL predictions** require re-running `01_efficientnet_feature_extraction_PAPER.ipynb` and
`02_efficientnet_fine_tuning_PAPER.ipynb` on GPU (TF 2.10 + CUDA 11.x), saving per-fold
predictions to `outputs/deep_learning/gloo/FE_XGB/fold_N/predictions.csv` and
`outputs/deep_learning/gloo/FT/fold_N/predictions.csv` respectively.
Once those files exist, this notebook runs fully without modification.

Each `predictions.csv` must contain columns: `m_id`, `day_of_study`, `y_true`, `y_proba`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

RADIOMICS_DIR = Path('outputs/radiomics/gloo/09_final_pipeline_PAPER/grouped_combinations_split_with_sampling')
DL_FE_DIR     = Path('outputs/deep_learning/gloo/FE_XGB')
DL_FT_DIR     = Path('outputs/deep_learning/gloo/FT')

## 1. Load Pooled Predictions

In [ ]:
def load_pooled_predictions(results_dir):
    """Concatenate per-fold prediction CSVs into one DataFrame."""
    dfs = []
    for fold_dir in sorted(Path(results_dir).glob('fold_*')):
        pred_file = fold_dir / 'predictions.csv'
        if pred_file.exists():
            dfs.append(pd.read_csv(pred_file))
    return pd.concat(dfs, ignore_index=True) if dfs else None

rad_df   = load_pooled_predictions(RADIOMICS_DIR)
dl_fe_df = load_pooled_predictions(DL_FE_DIR)
dl_ft_df = load_pooled_predictions(DL_FT_DIR)

print(f'Radiomics:  {len(rad_df) if rad_df is not None else "NOT FOUND"} exams')
print(f'DL FE+XGB: {len(dl_fe_df) if dl_fe_df is not None else "NOT FOUND — run 01_efficientnet_feature_extraction_PAPER.ipynb (GPU)"}')
print(f'DL FT:     {len(dl_ft_df) if dl_ft_df is not None else "NOT FOUND — run 02_efficientnet_fine_tuning_PAPER.ipynb (GPU)"}')

## 2. DeLong Test (exam-level)

DeLong et al. (1988) variance estimator for comparing two correlated AUCs obtained
on the same test observations.

In [ ]:
def delong_test(y_true, y_score1, y_score2):
    """
    DeLong test comparing two correlated AUCs.
    Returns: auc1, auc2, z_statistic, two-sided p-value
    """
    def structural_components(y_true, y_score):
        pos = y_score[y_true == 1]
        neg = y_score[y_true == 0]
        V10 = np.array([(p > neg).mean() + 0.5 * (p == neg).mean() for p in pos])
        V01 = np.array([(n < pos).mean() + 0.5 * (n == pos).mean() for n in neg])
        return V10.mean(), V10, V01

    auc1, V10_1, V01_1 = structural_components(y_true, y_score1)
    auc2, V10_2, V01_2 = structural_components(y_true, y_score2)
    n = (y_true == 1).sum()
    m = (y_true == 0).sum()

    var1   = np.var(V10_1, ddof=1) / n + np.var(V01_1, ddof=1) / m
    var2   = np.var(V10_2, ddof=1) / n + np.var(V01_2, ddof=1) / m
    cov12  = np.cov(V10_1, V10_2)[0, 1] / n + np.cov(V01_1, V01_2)[0, 1] / m
    var_d  = var1 + var2 - 2 * cov12

    if var_d <= 0:
        return auc1, auc2, np.nan, np.nan
    z = (auc1 - auc2) / np.sqrt(var_d)
    p = 2 * norm.sf(abs(z))
    return auc1, auc2, z, p


if rad_df is not None:
    print(f"Radiomics exam-level AUC: {roc_auc_score(rad_df.y_true, rad_df.y_proba):.4f}")

for name, dl_df in [('FE+XGB', dl_fe_df), ('FT', dl_ft_df)]:
    if dl_df is None or rad_df is None:
        print(f'DeLong Radiomics vs {name}: predictions not available.')
        continue
    merged = rad_df.merge(
        dl_df[['m_id', 'day_of_study', 'y_proba']].rename(columns={'y_proba': 'y_proba_dl'}),
        on=['m_id', 'day_of_study']
    )
    a1, a2, z, p = delong_test(merged.y_true.values, merged.y_proba.values, merged.y_proba_dl.values)
    sig = '(p<0.05)' if p is not None and p < 0.05 else '(not significant)'
    print(f'DeLong Radiomics vs {name}: AUC_rad={a1:.4f}, AUC_{name}={a2:.4f}, Z={z:.3f}, p={p:.4f} {sig}')

## 3. Subject-Level Permutation Test

With n=10 animals, the minimum achievable two-sided p-value is 1/C(10,5) ≈ 0.004.
Permutes animal-level labels and recomputes the difference in animal-level AUC.

In [ ]:
def subject_permutation_test(y_true_a, proba1_a, proba2_a, n_perm=10000, seed=42):
    """
    Permutation test for AUC difference (model1 - model2) at animal level.
    Permutes animal labels; robust to the small n.
    """
    rng = np.random.default_rng(seed)
    obs = roc_auc_score(y_true_a, proba1_a) - roc_auc_score(y_true_a, proba2_a)
    null = []
    for _ in range(n_perm):
        yp = rng.permutation(y_true_a)
        if len(np.unique(yp)) < 2:
            continue
        null.append(roc_auc_score(yp, proba1_a) - roc_auc_score(yp, proba2_a))
    p = (np.abs(null) >= abs(obs)).mean()
    return obs, p


if rad_df is not None:
    rad_a = rad_df.groupby('m_id').agg(y_true=('y_true', 'first'), y_proba=('y_proba', 'mean')).reset_index()
    print(f'Radiomics animal-level AUC: {roc_auc_score(rad_a.y_true, rad_a.y_proba):.4f}')

    for name, dl_df in [('FE+XGB', dl_fe_df), ('FT', dl_ft_df)]:
        if dl_df is None:
            print(f'Permutation test vs {name}: DL predictions not available.')
            continue
        dl_a = dl_df.groupby('m_id').agg(y_proba_dl=('y_proba', 'mean')).reset_index()
        merged_a = rad_a.merge(dl_a, on='m_id')
        diff, p = subject_permutation_test(merged_a.y_true.values, merged_a.y_proba.values, merged_a.y_proba_dl.values)
        print(f'Permutation test Radiomics vs {name} (n=10 animals): AUC_diff={diff:.4f}, p={p:.4f}')

## 4. Interpretation Note

With n=10 independent subjects, both tests have limited statistical power. A non-significant
result does not establish equivalence; it reflects that reassigning a single animal across folds
can substantially change the observed AUC difference. Results should be reported alongside
the bootstrap confidence intervals computed in `09_final_pipeline_PAPER.ipynb`.